# ESG sentence classification on Kaggle GPU

Classifies Vietnamese/English annual-report sentences with **`nguyen599/ViDeBERTa-v3-ESG-base`** (DeBERTa-v2, multi-label).

**Before running:**
1. Sidebar &rarr; **Notebook options** &rarr; *Accelerator* = **GPU T4** (NOT P100 — the image's PyTorch dropped P100/sm_60 support), *Internet* = **On** (needed to download the model the first time).
2. **Add data**: upload `aaa_sentences.jsonl` (produced locally by `python -m data_processing.prepare_sentences ...`) as a Kaggle **Dataset**, then attach it. It mounts at `/kaggle/input/<your-dataset-slug>/`.
3. Set `SENTENCES_PATH` in the config cell to that file.

**Why sigmoid, not softmax:** the model's `config.problem_type == "multi_label_classification"`, so each label (Environmental / Social / Governance / Neutral) gets an independent sigmoid score.

**How labels are decided (tuned on the AAA report):**
- A sentence is tagged with a pillar when that pillar's score &ge; `THRESHOLD` (**0.45**).
- A sentence is **ESG-relevant** when `Neutral < NEUTRAL_THRESHOLD` (**0.5**), *not* when a single pillar clears the bar. In practice this model spreads probability like a softmax and rarely tags two pillars at once, so a clearly-ESG sentence can have its signal split across two pillars (e.g. labour-law compliance &rarr; S=0.44, G=0.44) with neither clearing the threshold. Keying `esg` off Neutral recovers those cases.

Output (downloadable from `/kaggle/working/`): `classified.jsonl` and `classified.csv`.

In [ ]:
# 1) Dependencies. torch + CUDA already ship on Kaggle GPU images.
!pip install -q "transformers==4.51.0" sentencepiece protobuf

In [ ]:
# 2) Config -- edit SENTENCES_PATH to match your attached dataset.
import glob, os

MODEL_ID     = "nguyen599/ViDeBERTa-v3-ESG-base"
SENTENCES_PATH = "/kaggle/input/aaa-esg-sentences/aaa_sentences.jsonl"  # <-- EDIT ME
OUT_JSONL    = "/kaggle/working/classified.jsonl"
OUT_CSV      = "/kaggle/working/classified.csv"
BATCH_SIZE   = 64
MAX_LENGTH   = 256   # sentences are short; model cap is 512
THRESHOLD    = 0.45  # sigmoid cutoff for assigning an E/S/G pillar tag
NEUTRAL_THRESHOLD = 0.5  # ESG-relevant when the Neutral score is below this
ESG_PILLARS  = ("Environmental", "Social", "Governance")

# Auto-locate the file if the slug differs from the guess above.
if not os.path.exists(SENTENCES_PATH):
    hits = glob.glob("/kaggle/input/**/*.jsonl", recursive=True) + \
           glob.glob("/kaggle/input/**/*.csv", recursive=True)
    assert hits, "No .jsonl/.csv found under /kaggle/input -- attach your dataset."
    SENTENCES_PATH = hits[0]
    print("Using auto-detected input:", SENTENCES_PATH)

In [ ]:
# 3) Load the model on GPU.
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()

def _norm(lbl):
    return "Neutral" if lbl.strip().lower() in {"neural", "neutral", "none"} else lbl

id2label = {int(i): _norm(l) for i, l in model.config.id2label.items()}
print("problem_type:", model.config.problem_type, "| labels:", id2label)

In [ ]:
# 4) Read sentences.
import csv, json

def read_sentences(path):
    if path.lower().endswith(".csv"):
        with open(path, encoding="utf-8-sig", newline="") as f:
            return list(csv.DictReader(f))
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

rows = read_sentences(SENTENCES_PATH)
texts = [str(r.get("text", "")) for r in rows]
print(f"{len(texts)} sentences loaded.")

In [ ]:
# 5) Batched multi-label inference (sigmoid).
#    Pillar tags = E/S/G scores >= THRESHOLD.
#    esg flag    = Neutral < NEUTRAL_THRESHOLD (robust to signal split across pillars).
preds = []
for start in range(0, len(texts), BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]
    enc = tokenizer(batch, padding=True, truncation=True,
                    max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits).cpu().tolist()
    for row in probs:
        scores = {id2label[i]: round(float(p), 4) for i, p in enumerate(row)}
        labels = [p for p in ESG_PILLARS if scores.get(p, 0.0) >= THRESHOLD]
        esg = scores.get("Neutral", 0.0) < NEUTRAL_THRESHOLD
        preds.append({"scores": scores, "labels": labels, "esg": esg})

merged = [{**r, **p} for r, p in zip(rows, preds)]
print("Done. Example:", json.dumps(merged[0], ensure_ascii=False)[:300])

In [ ]:
# 6) Write outputs + summary.
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in merged:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

fields = ["source_pdf", "page", "sentence_index", "text",
          "labels", "esg", *ESG_PILLARS, "Neutral"]
with open(OUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    for r in merged:
        s = r.get("scores", {})
        w.writerow({**{k: r.get(k, "") for k in ("source_pdf", "page", "sentence_index", "text")},
                    "labels": ", ".join(r.get("labels", [])), "esg": r.get("esg", ""),
                    **{p: s.get(p, "") for p in (*ESG_PILLARS, "Neutral")}})

counts = {l: 0 for l in ESG_PILLARS}
esg_total = 0
for r in merged:
    for l in r["labels"]:
        counts[l] = counts.get(l, 0) + 1
    esg_total += int(r["esg"])
print(f"Wrote {OUT_JSONL} and {OUT_CSV}")
print(f"ESG-relevant (Neutral<{NEUTRAL_THRESHOLD}): {esg_total}/{len(merged)} | pillar tag counts: {counts}")

In [ ]:
# 7) (Optional) eyeball a few results per pillar.
import itertools
for pillar in ESG_PILLARS:
    print(f"\n=== {pillar} ===")
    sample = (r for r in merged if pillar in r["labels"])
    for r in itertools.islice(sample, 3):
        print(f"  p{r.get('page')} [{r['scores'][pillar]:.2f}] {r['text'][:120]}")